<a href="https://colab.research.google.com/github/DavidQuinonez4/Analitica-de-Negocios/blob/main/P_Anal%C3%ADtica_DavidQuinonez.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#*Caso de Estudio:*

Una entidad del sector salud busca desarrollar un modelo que permita clasificar a los pacientes según la probabilidad de presentar diabetes a partir de diferentes variables relacionadas con su estado de salud. Para ello, la organización cuenta con la base de datos llamada *diabetes.csv*, la cual contiene información clínica relevante de los pacientes.

Para abordar este problema de una manera eficiente, implementaremos diferentes técnicas de analítica de datos, entre ellas Naive Bayes, Árboles de Decisión y Clustering mediante K-Means. Inicialmente se realizará un análisis de correlación con el objetivo de identificar qué variables presentan mayor relación entre sí y, especialmente, cuáles tienen una mayor asociación con la variable de salida (Outcome), que indica la presencia o ausencia de diabetes.

La variables que tendremos en cuenta en la realización de este análisis son las siguientes:

Pregnancies = Número de veces que la paciente ha estado embarazada.

Glucose = Nivel de glucosa en la sangre después de realizar una prueba oral de tolerancia a la glucosa.

BloodPressure = Presión arterial diastólica.

SkinThickness = Grosor del pliegue de la piel en el tríceps, utilizado para estimar la grasa corporal.

Insulin = Nivel de insulina en sangre (dos horas después de la prueba de glucosa).

BMI = Relaciona el peso con la estatura y permite identificar posibles casos de sobrepeso u obesidad.

DiabetesPedigreeFunction = Mide la probabilidad genética de que una persona desarrolle diabetes según su historial familiar.

Age = Indica la edad de la persona en años.

Outcome = Resultado del diagnóstico: 0 = no tiene diabetes, 1 = tiene diabetes.





0. Cargamos librerías de trabajo

In [ ]:
import numpy as np
import pandas as pd

#Librerias especificas Naive Bayes
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import confusion_matrix

#Librerias especificas árboles de decision
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix

#Librerias especificas K-Means
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.metrics import confusion_matrix

1. Cargamos base de datos para el trabajo

In [ ]:
nxl='/content/diabetes.xlsx'
XDB=pd.read_excel(nxl,sheet_name=0)
XD=XDB.iloc[:,8]
yd=XDB.iloc[:,8]

2. Procedemos con la correlación

In [ ]:
# Visualización la matriz de correlación
correlation_matrix_all = XDB.corr()
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix_all, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Matriz de correlación con todas las variables')
plt.show()

*Análisis de la Correlación:*

A partir del heatmap de correlaciones podemos identificar algunas relaciones importantes entre las variables del dataset. La correlación más alta entre variables independientes se presenta entre Pregnancies y Age, con un valor de 0.54. Esto nos indica que, a mayor edad, es más probable que la persona haya tenido un mayor número de embarazos, lo cual resulta lógico porque con el paso del tiempo aumenta la posibilidad de haber estado embarazada en más ocasiones.También podemos observar una relación entre SkinThickness e Insulin de aproximadamente 0.44. Esto sugiere que cuando el grosor del pliegue cutáneo es mayor, los niveles de insulina también tienden a incrementarse, lo que puede estar relacionado con mayores niveles de grasa corporal.

También identificamos una correlación moderada entre BMI y SkinThickness con un valor cercano a 0.39. Esto indica que cuando el índice de masa corporal aumenta, el espesor del pliegue cutáneo también tiende a aumentar, reflejando una relación con el nivel de grasa corporal.

Al analizar la relación con la variable de salida (Outcome), podemos ver que la variable con mayor correlación es Glucose, con un valor de 0.47. Esto nos muestra que niveles elevados de glucosa en sangre están fuertemente asociados con la presencia de diabetes.

Otras variables como BMI (0.29) y Age (0.24) también presentan cierta relación con el resultado, lo que nos sugiere que un mayor índice de masa corporal y una mayor edad pueden incrementar la probabilidad de desarrollar diabetes.

Es decir que el nivel de glucosa es la variable que más se relaciona con la presencia de diabetes, seguida por factores como el BMI y la edad, mientras que las demás variables muestran correlaciones más bajas con el resultado.

3. Procedemos con Variables de Trabajo para Modelo Naive Bayes






In [ ]:
XD=XDB[["Pregnancies","Glucose", "BloodPressure","SkinThickness", "Insulin", "BMI", "DiabetesPedigreeFunction", "Age"]] #Variables de trabajo
XDB.head(10)
yd=XDB[["Outcome"]]
yd.head()

4. Implementación del modelo Naive Bayes

In [ ]:
from sklearn.model_selection import train_test_split

X_train_nb, X_test_nb, y_train_nb, y_test_nb = train_test_split(XD, yd, test_size=0.3, random_state=42)

mnb=GaussianNB()
mnb.fit(X_train_nb, y_train_nb.values.ravel()) #Variables de entrada y la salida con datos de entrenamiento

#Mostrar medias
u=mnb.theta_
sigma=mnb.var_;sigma=np.sqrt(sigma)
print("Medias (theta_) para cada clase:\n", u)
print ("Desviaciones estándar (sigma_) para cada clase:\n", sigma)

# Imprimimos los nombres de las características
feature_names = XD.columns.tolist()
print("Orden de las características:\n", feature_names)

5. Procedemos con la evaluación del modelo utilizando la matriz de confusión

In [ ]:
ydp=mnb.predict(X_test_nb) # Realizar predicciones en el conjunto de prueba
cm=confusion_matrix(y_test_nb, ydp)
print("Matriz de Confusión:\n", cm)

#Se determinan las métricas de la matriz de confusión
VN=cm[0,0]; FP=cm[0,1]; FN=cm[1,0]; VP=cm[1,1]; TDatos=len(y_test_nb)

#1. Exactitud: Funcionamiento general del modelo
Ex=(VP+VN)/TDatos
print("Exactitud=",Ex)

#2. Tasa Error: %Fallos del modelo
TE=(FP+FN)/TDatos
print("Tasa Error=",TE)

#3.Sensibilidad: Cómo se comportó con respectos a los Verdaderos Positivos
Se=VP/(VP+FN)
print("Sensibilidad:", Se)

#4.Especificidad: Cómo se comporta pronosticando negativos
Es=VN/(VN+FP)
print("Especificidad:", Es)

#5.Precisión: Es una versión de cómo se comporta el modelo frente a los positivos solamente

Pos=VP/(VP+FP)
print("Precision:", Pos)

#6.Prediccion Negativa: Cómo funciona el modelo pronosticando negativos (en este caso enfermos)

Neg=VN/(VN+FN)
print("Prediccion Negativa:", Neg)

6. Procedemos a la evaluación de uno de los pacientes

In [ ]:
XDp=[2,85,65,29,94,39.6,0.93,27] # Esta lista cuenta con 8 características, coincide con XD
ydc=mnb.predict([XDp])
print(f"El modelo predice: {ydc}")

if ydc[0] == 1:
    print("El paciente es: Diabético")
else:
    print("El paciente es: No Diabético")

*Análisis de Resultado del Modelo Naive Bayes*

A partir de los resultados del modelo podemos comparar las medias (theta) y desviaciones estándar (sigma) de cada variable para los dos grupos: pacientes sin diabetes (0) y pacientes con diabetes (1). Esto permite identificar cómo cambian los valores de las variables según la presencia de la enfermedad.

Al analizar las medias, podemos observar que las personas con diabetes presentan valores más altos en variables como Glucose (141.9 vs 109.5), BMI (35.33 vs 30.16), Insulin (101.05 vs 68.40) y Age (37.52 vs 30.67). Esto indica que los pacientes con diabetes tienden a tener niveles más elevados de glucosa, mayor índice de masa corporal, mayor insulina y una edad promedio más alta, lo cual coincide con algunos factores de riesgo conocidos para esta enfermedad.

En la matriz de confusión, el modelo clasificó correctamente 119 pacientes sin diabetes y 53 pacientes con diabetes, mientras que cometió 32 falsos positivos y 27 falsos negativos.

La exactitud del modelo es de 74.45%, lo que significa que aproximadamente tres de cada cuatro predicciones fueron correctas. La sensibilidad (0.6625) muestra que el modelo identifica cerca del 66% de los pacientes con diabetes, mientras que la especificidad (0.788) indica que reconoce correctamente alrededor del 79% de los pacientes sin la enfermedad.

Finalmente, al evaluar un paciente con el modelo, la predicción obtenida fue [0], lo que significa que el modelo lo clasifica como no diabético, de acuerdo con los patrones aprendidos a partir de los datos, es decir que podemos observar que el modelo tiene un desempeño aceptable, aunque muestra una mayor capacidad para identificar correctamente a los pacientes sin diabetes que a aquellos que sí la presentan.

7. Procedemos con el Modelo de Árbol de Decisión


In [ ]:
#Cargamos datos de trabajo
nxl='/content/diabetes.xlsx'
XDB=pd.read_excel(nxl,sheet_name=0)
XD=XDB.iloc[:,8]
yd=XDB.iloc[:,8]
display(XD)

8. Procedemos con la implementación del árbol

In [ ]:
nxl='/content/diabetes.xlsx'
XDB=pd.read_excel(nxl,sheet_name=0)
XD=XDB.drop('Outcome', axis=1)
yd=XDB['Outcome']
display(XD.head())

In [ ]:
mar=DecisionTreeClassifier(criterion='gini',max_depth=4)
mar.fit(XD,yd) #Aquí el modelo busca la relacion entrada?/salida

#¿Qué función cumplió el modelo?

ydp=mar.predict(XD) # Esto es lo que pronóstica el modelo
#Se construye la matriz de confusión
cm=confusion_matrix(yd,ydp)
display(cm)
VN=cm[0,0]; FP=cm[0,1];FN=cm[1,0]; VP=cm[1,1]

#metricas de desempeño
exactitud=(VP+VN)/(VP+VN+FP+FN) #1.comportamiento general
print('La exactitud es:',exactitud)
sensibilidad=VP/(VP+FN) #2.Cómo se comporta pronosticando positivos
print('La sensibilidad es:',sensibilidad)
especificidad=VN/(VN+FP) #3. Cómo se comporta frente a los negativos
print('La especificidad es:',especificidad)
precision=VP/(VP+FP) #4. Cómo se comporta pronosticando positivos
print('La precision es:',precision)
preneg=VN/(VN+FN) #5. Cómo se comporta pronosticando negativos
print('La prenegatividad es:',preneg)

9. Despliegue del Árbol de Decisión

In [ ]:
from sklearn.tree import export_graphviz # exporta los datos a un grafico
from pydotplus import graph_from_dot_data # es un graficador

vs=["Pregnancies","Glucose", "BloodPressure","SkinThickness", "Insulin", "BMI", "DiabetesPedigreeFunction", "Age"] #titulos del arbol

dot_data=export_graphviz(mar,feature_names=vs) #exportar de numeros a graficos en pdf
graph=graph_from_dot_data(dot_data) #hacemos el grafico
graph.write_png('arbol.png')

*Análisis de Resultado del Árbol de Decisión

Al analizar el árbol de decisión podemos notar que la variable que primero separa los datos es Glucose ≤ 127.5, lo que nos indica que el nivel de glucosa es el factor más determinante para diferenciar entre pacientes con y sin diabetes. En el nodo inicial se incluyen 768 observaciones, de las cuales 500 corresponden a pacientes no diabéticos y 268 a diabéticos, con un Gini de 0.454, lo que refleja que en ese punto todavía existe mezcla entre ambas clases.

A medida que el árbol se desarrolla, podemos observar que el modelo utiliza principalmente Age, BMI, Insulin y nuevamente Glucose para seguir dividiendo a los pacientes. Por ejemplo, cuando Glucose es menor o igual a 127.5 y Age es menor o igual a 28.5, el modelo analiza el BMI, lo que sugiere que el índice de masa corporal se vuelve relevante para clasificar a pacientes jóvenes. En cambio, cuando Glucose supera 127.5, el modelo considera variables como BMI e Insulin para mejorar la clasificación.

También podemos identificar alrededor de 6 nodos puros (gini = 0), lo que significa que en esos puntos todos los datos pertenecen a una sola clase. El nodo puro con mayor número de observaciones contiene 39 casos con valor [39,0], lo que indica que todos los pacientes de ese nodo fueron clasificados como no diabéticos. La regla asociada sería: si Glucose ≤ 127.5, Age ≤ 28.5 y BMI ≤ 26.35, entonces el paciente se clasifica como No Diabético, lo que sugiere que pacientes jóvenes con niveles bajos de glucosa y un BMI bajo presentan una probabilidad muy baja de tener diabetes.

En general, podemos concluir que Glucose, BMI, Age e Insulin son las variables que más influyen en la construcción del árbol. Además, este modelo permite generar reglas de decisión claras y fáciles de interpretar, lo cual facilita entender bajo qué condiciones se clasifica a un paciente como diabético o no.

10. Modelo de Clusterización K-Means

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix_all, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Matriz de Correlación de todas las variables')
plt.show()

11. Se procede con la implementación del Modelo KMeans

In [ ]:
np.random.seed(42) #Esto permite generar las mismas semillas para todos
NC=5# 5 segmentos o perfiles de pacientes

XD_features = XDB[["Pregnancies","Glucose", "BloodPressure","SkinThickness", "Insulin", "BMI", "DiabetesPedigreeFunction", "Age"]]

mkm=KMeans(n_clusters=NC,random_state=42, n_init=10)
mkm.fit(XD_features) # El modelo solo busca patrones en los datos de entrada (XD)

#Obtenemos las caracteristicas de cada grupo
#Representa los perfiles de las personas mkm.cluster_centers_ que encontró el modelo
tabla=pd.DataFrame(mkm.cluster_centers_,columns=XD_features.columns)
display(tabla)

#Para saber los porcentajes de Positivos o Negativos por segmento, sucursal o perfil
ydp=mkm.labels_ #Esto me indica en que cluster esta clasificado cada dato
NDc=np.bincount(ydp) #Número de datos por segmento
print("El número de datos por cluster es:",NDc)

# Añadimos la columna del cluster al DataFrame original para el análisis posterior
XDB['Cluster'] = ydp

# Aquí calculamos la probabilidad de diabetes para cada cluster
# La variable 'Outcome' es 1 para diabetes y 0 para no diabetes
diabetes_probability = XDB.groupby('Cluster')['Outcome'].mean()
print("\nProbabilidad de diabetes por cluster:")
print(diabetes_probability)

12. Porcentaje de pacientes enfermos con diabetes

In [ ]:
Posit_counts = XDB.groupby('Cluster').agg({'Outcome': lambda x: (x == 1).sum()}).rename(columns={'Outcome': 'Diabetes_Count'})
Neg_counts = XDB.groupby('Cluster').agg({'Outcome': lambda x: (x == 0).sum()}).rename(columns={'Outcome': 'No_Diabetes_Count'})
counts_by_cluster = Posit_counts.join(Neg_counts)
counts_by_cluster['Total_Patients'] = counts_by_cluster['Diabetes_Count'] + counts_by_cluster['No_Diabetes_Count']

counts_by_cluster['Prob_Diabetes'] = counts_by_cluster['Diabetes_Count'] / counts_by_cluster['Total_Patients']
counts_by_cluster['Prob_No_Diabetes'] = counts_by_cluster['No_Diabetes_Count'] / counts_by_cluster['Total_Patients']

print("Número de pacientes con y sin diabetes por cluster:")
display(counts_by_cluster)

print("\nProbabilidad de diabetes y no diabetes por cluster (en formato similar a 'PreApr'/'PreNeg'):")
df2 = counts_by_cluster[['Prob_Diabetes', 'Prob_No_Diabetes']]
df2.columns = ['Diabetes_Prob', 'No_Diabetes_Prob']
display(df2)

13. Procedemos a analizar a uno de los pacientes

In [ ]:
import warnings
warnings.filterwarnings('ignore')
nueva_persona=np.array([[2,85,65,29,94,39.6,0.93,27]])
ydp=mkm.predict(nueva_persona)
NCl=ydp [0]#esto indica el cluster al que pertenece una persona
print("El paciente pertenece al cluster:",ydp)
# Accediendo a las probabilidades desde df2
print("El porcentaje de Diabetes para el nuevo paciente es:",df2['Diabetes_Prob'].iloc[NCl])
print("El porcentaje de No Diabetes para el nuevo paciente es:",df2['No_Diabetes_Prob'].iloc[NCl])

14. Se procede con la gráfica de lso clusters utilizando las variables Glucosa y Presión en Sangre

In [ ]:
centers=mkm.cluster_centers_ #Centroides, Perfil y Segmento

ing=np.array(XD_features.iloc[:,0]); egr=np.array(XD_features.iloc[:,7])

plt.figure(figsize=(10, 8))
plt.scatter(ing,egr,c=mkm.labels_,cmap='rainbow')
plt.scatter(centers[:,0],centers[:,7],c='black',s=2000,alpha=0.5)

plt.xlabel(XD_features.columns[0]) # 'Pregnancies'
plt.ylabel(XD_features.columns[7]) # 'Age'
plt.title('Clusters de Pacientes (Pregnancies vs Age)')

#ADD NUMBERS
for i, center in enumerate(centers):
    plt.text(center[0],center[7],str(i),color='white', fontsize=12, ha='center', va='center')
plt.show()

In [ ]:
centers=mkm.cluster_centers_ #Centroides, Perfil y Segmento

glucose=np.array(XD.iloc[:,1])
bmi=np.array(XD.iloc[:,5])

plt.figure()
plt.scatter(glucose,bmi,c=mkm.labels_,cmap='rainbow')
plt.scatter(centers[:,1],centers[:,5],c='black',s=2000,alpha=0.5)

plt.xlabel("Glucose")
plt.ylabel("BMI")

for i, c in enumerate(centers):
    plt.text(c[1],c[5], str(i), color='white', fontsize=12, ha='center', va='center')

plt.title("Clusters de Pacientes (Glucose vs BMI)")
plt.show()

*Análisis de Resultado Modelo K-Means*

Al observar el heatmap de correlación podemos notar algunas relaciones importantes entre las variables. La más alta aparece entre Pregnancies y Age (0.54), lo que sugiere que a mayor edad es más probable haber tenido más embarazos. También vemos relación entre SkinThickness e Insulin (0.44) y entre BMI y SkinThickness (0.39), indicando que variables asociadas con grasa corporal tienden a aumentar juntas. Respecto a Outcome, la variable más relacionada es Glucose (0.47), seguida por BMI (0.29) y Age (0.24).

Aquí observamos que el modelo K-Means agrupó a los pacientes en 5 clusters: cluster 0 (204), cluster 1 (198), cluster 2 (295), cluster 3 (8) y cluster 4 (63). El cluster 4 presenta la mayor probabilidad de diabetes (53.96%), por lo que representa el grupo de mayor riesgo. El cluster 1 también muestra una probabilidad relativamente alta (42.42%), mientras que el cluster 2 refleja un nivel intermedio (32.54%).

Por el contrario, el cluster 0 presenta una probabilidad menor (25%), lo que indica un grupo con menor riesgo. El cluster 3 es el más pequeño con 8 pacientes y una probabilidad de 37.5%, por lo que podría representar un grupo particular dentro de los datos.

Al evaluar un nuevo paciente, observamos que el modelo la ubica en el cluster 0, donde la probabilidad estimada de diabetes es 25%, lo que sugiere un riesgo relativamente bajo según sus características. En general, este modelo nos permite identificar distintos perfiles de pacientes y diferenciar grupos con mayor o menor probabilidad de diabetes.

*Conclusión General:*

En general, al comparar los tres modelos podemos observar que todos permiten analizar la presencia de diabetes desde diferentes enfoques. Naive Bayes ofrece una buena capacidad de clasificación basada en probabilidades, el árbol de decisión permite entender fácilmente las reglas que influyen en el diagnóstico, y K-Means ayuda a identificar distintos perfiles de pacientes según sus características. En conjunto, estos modelos permiten comprender mejor los factores asociados con la diabetes y apoyar procesos de análisis y toma de decisiones en el ámbito de la salud.